# 02b — Traduction Google Texte avec Playwright

Ce notebook pilote un navigateur **visible** pour traduire progressivement les partitions français → baoulé dans le mode Texte. Google utilise le code `bci` dans l'URL de son interface pour le baoulé. Le pilote de 10 blocs a été validé avant l'activation du mode complet.

Principes de sécurité et de reproductibilité : une seule session, débit fixe et modéré, sauvegarde après chaque bloc, reprise automatique, aucune rotation de proxy ou d'identité, aucun contournement de CAPTCHA, et arrêt immédiat si Google affiche une protection.

> Cette interface grand public n'est pas une API. Vérifiez que cet usage respecte les conditions applicables à votre organisation avant de dépasser le pilote.

In [1]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import asyncio
import hashlib
import json
import os
import re
import subprocess
import sys
import threading
import time
from urllib.parse import urlencode

PROJECT_ROOT = Path.cwd()
SPLIT_DIR = PROJECT_ROOT / "data" / "01_splits"
PIPELINE_DIRECTION = os.environ.get("LREE_PIPELINE_DIRECTION", "fr_bau")
if PIPELINE_DIRECTION not in {"fr_bau", "bau_bau"}:
    raise ValueError(f"Direction inconnue : {PIPELINE_DIRECTION}")
default_step_name = "02b_text_browser" if PIPELINE_DIRECTION == "fr_bau" else "02c_bau_bau_browser"
STEP_DIR = Path(os.environ.get("LREE_STEP_DIR", str(PROJECT_ROOT / "data" / default_step_name)))
STEP_DIR.mkdir(parents=True, exist_ok=True)

UNITS_PATH = STEP_DIR / "translation_units.jsonl"
BATCHES_PATH = STEP_DIR / "translation_batches.jsonl"
RESULTS_PATH = STEP_DIR / "browser_results.jsonl"
PARTIAL_TRANSLATIONS_PATH = STEP_DIR / "partial_translations.jsonl"
VALIDATION_REPORT_PATH = STEP_DIR / "validation_report.json"
LOCAL_BROWSER_LIB_DIR = PROJECT_ROOT / ".playwright-libs" / "usr" / "lib" / "x86_64-linux-gnu"

TARGET_LANGUAGE_CODE = "bci"
MAX_BATCH_CHARACTERS = 3_500  # garde aussi l'URL encodée sous une taille fiable
CHECKPOINT_EVERY_BATCHES = 100  # parser/contrôler, puis continuer automatiquement
DELAY_SECONDS = float(os.environ.get("LREE_DELAY_SECONDS", "16.0"))  # 2 notebooks parallèles = débit agrégé prudent
INITIAL_DELAY_SECONDS = float(os.environ.get("LREE_INITIAL_DELAY_SECONDS", "0"))
TRANSLATION_TIMEOUT_SECONDS = 45
RUN_BROWSER = True           # passer à True après lecture de la procédure
BROWSER_CHANNEL = None     # utiliser le Chromium installé et géré par Playwright

print("Répertoire de travail :", STEP_DIR)
print("Direction              :", PIPELINE_DIRECTION)
print("Code cible Google     :", TARGET_LANGUAGE_CODE)
print("Contrôle automatique tous les blocs :", CHECKPOINT_EVERY_BATCHES)

Répertoire de travail : /home/vedem_worker/scrapt-bau/data/02b_text_browser
Direction              : fr_bau
Code cible Google     : bci
Contrôle automatique tous les blocs : 100


## 1. Construire les unités de traduction

Pour `fr_bau`, seule la réponse est traduite. Pour `bau_bau`, l'instruction, le contexte non vide et la réponse sont traduits. Les textes exceptionnellement longs sont découpés à une frontière naturelle puis réassemblés après traduction.

In [2]:
def read_jsonl(path):
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def write_jsonl_atomic(path, records):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="\n") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    temporary.replace(path)

ALL_FIELD_MAP = {
    "fr_bau": [("output_fr", "output_bau")],
    "bau_bau": [
        ("instruction_fr", "instruction_bau"),
        ("input_fr", "input_bau"),
        ("output_fr", "output_bau"),
    ],
}
FIELD_MAP = {PIPELINE_DIRECTION: ALL_FIELD_MAP[PIPELINE_DIRECTION]}

source_partitions = {
    direction: read_jsonl(SPLIT_DIR / f"{direction}_source.jsonl")
    for direction in FIELD_MAP
}

def split_long_text(text, max_chars):
    if len(text) <= max_chars:
        return [text]
    chunks = []
    remaining = text
    while len(remaining) > max_chars:
        window = remaining[:max_chars + 1]
        candidates = [
            window.rfind("\n\n"),
            window.rfind("\n"),
            window.rfind(". "),
            window.rfind("! "),
            window.rfind("? "),
            window.rfind("; "),
            window.rfind(", "),
            window.rfind(" "),
        ]
        cut = max(candidates)
        if cut < max_chars // 2:
            cut = max_chars
        else:
            cut += 1
        chunks.append(remaining[:cut].strip())
        remaining = remaining[cut:].strip()
    if remaining:
        chunks.append(remaining)
    return chunks

units = []
max_payload_chars = MAX_BATCH_CHARACTERS - 80
unit_number = 0
for direction, records in source_partitions.items():
    for record in records:
        for source_field, target_field in FIELD_MAP[direction]:
            source_text = record[source_field]
            if not source_text:
                continue
            parts = split_long_text(source_text, max_payload_chars)
            for part_index, part_text in enumerate(parts):
                unit_number += 1
                units.append({
                    "unit_id": f"u{unit_number:08d}",
                    "record_id": record["id"],
                    "source_row_index": record["source_row_index"],
                    "direction": direction,
                    "source_field": source_field,
                    "target_field": target_field,
                    "part_index": part_index,
                    "part_count": len(parts),
                    "source_text": part_text,
                    "source_sha256": hashlib.sha256(part_text.encode("utf-8")).hexdigest(),
                })

write_jsonl_atomic(UNITS_PATH, units)
print(f"Unités créées : {len(units):,}")
print(f"Fichier         : {UNITS_PATH}")

Unités créées : 22,009
Fichier         : /home/vedem_worker/scrapt-bau/data/02b_text_browser/translation_units.jsonl


## 2. Regrouper les unités en blocs identifiables

Chaque fragment commence par un identifiant numérique de huit chiffres. Une traduction n'est acceptée que si tous les identifiants du bloc sont encore présents dans le résultat.

In [3]:
def serialize_unit(unit):
    numeric_id = unit["unit_id"][1:]
    return f"{numeric_id}: {unit['source_text']}"

unit_by_id = {unit["unit_id"]: unit for unit in units}

batches = []
current_units = []
current_parts = []
current_length = 0

def flush_batch():
    global current_units, current_parts, current_length
    if not current_units:
        return
    batch_number = len(batches) + 1
    source_text = "\n".join(current_parts)
    batches.append({
        "batch_id": f"batch_{batch_number:05d}",
        "unit_ids": [unit["unit_id"] for unit in current_units],
        "source_text": source_text,
        "source_characters": len(source_text),
        "source_sha256": hashlib.sha256(source_text.encode("utf-8")).hexdigest(),
    })
    current_units, current_parts, current_length = [], [], 0

for unit in units:
    serialized = serialize_unit(unit)
    separator_length = 1 if current_parts else 0
    if current_parts and current_length + separator_length + len(serialized) > MAX_BATCH_CHARACTERS:
        flush_batch()
    current_units.append(unit)
    current_parts.append(serialized)
    current_length += separator_length + len(serialized)
flush_batch()

assert all(batch["source_characters"] <= MAX_BATCH_CHARACTERS for batch in batches)
assert sum(len(batch["unit_ids"]) for batch in batches) == len(units)
write_jsonl_atomic(BATCHES_PATH, batches)

total_characters = sum(batch["source_characters"] for batch in batches)
print(f"Blocs préparés       : {len(batches):,}")
print(f"Caractères avec identifiants : {total_characters:,}")
print(f"Contrôle automatique tous les : {CHECKPOINT_EVERY_BATCHES} blocs")
print(f"Attente minimale entre contrôles : {CHECKPOINT_EVERY_BATCHES * DELAY_SECONDS / 60:.1f} minutes")

Blocs préparés       : 1,936
Caractères avec identifiants : 6,321,511
Contrôle automatique tous les : 100 blocs
Attente minimale entre contrôles : 26.7 minutes


## 3. Installer Playwright

Exécutez cette cellule dans un environnement possédant une interface graphique. Elle installe la bibliothèque Python puis le Chromium compatible géré par Playwright. L'installation du navigateur est ignorée automatiquement s'il est déjà présent.

In [4]:
try:
    import playwright
    print("Playwright est déjà installé.")
except ImportError:
    print("Installation de Playwright…")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "playwright"])
    import playwright
    print("Playwright installé.")

if BROWSER_CHANNEL is None:
    print("Vérification/installation du navigateur Chromium compatible…")
    subprocess.check_call([sys.executable, "-m", "playwright", "install", "chromium"])
    print("Chromium Playwright est prêt.")

Playwright est déjà installé.
Vérification/installation du navigateur Chromium compatible…
Chromium Playwright est prêt.


In [ ]:
import os
from pathlib import Path

local_libs = (
    Path.cwd()
    / ".playwright-libs"
    / "usr"
    / "lib"
    / "x86_64-linux-gnu"
)

previous = os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = (
    str(local_libs) + (":" + previous if previous else "")
)

print("Bibliothèques Chromium :", local_libs)
print("Existe :", local_libs.exists())

## 4. Exécuter la traduction continue dans un navigateur visible

Avec `RUN_BROWSER = True`, le navigateur s'ouvre sur Google Traduction. Acceptez ou refusez les cookies si nécessaire, vérifiez visuellement **Français → Baoulé**, puis revenez au notebook et appuyez sur Entrée. Une seule exécution traite tous les blocs encore manquants.

Ne touchez plus à la page pendant l'exécution. Chaque bloc est transmis par une URL `text=` puis lu directement dans le panneau de résultat. Le notebook sauvegarde immédiatement chaque succès, reconstruit et contrôle automatiquement le dataset tous les 100 blocs, puis continue. Il s'arrête uniquement à la fin, sur une protection Google ou sur une erreur réelle.

In [5]:
class ProtectionDetected(RuntimeError):
    pass

def append_jsonl(path, record):
    with path.open("a", encoding="utf-8", newline="\n") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()

def completed_batch_ids():
    if not RESULTS_PATH.exists():
        return set()
    return {record["batch_id"] for record in read_jsonl(RESULTS_PATH) if record.get("status") == "ok"}

def expected_markers(batch):
    return [f"{unit_id[1:]}:" for unit_id in batch["unit_ids"]]

def make_sub_batch(parent_batch, unit_ids, suffix):
    source_text = "\n".join(serialize_unit(unit_by_id[unit_id]) for unit_id in unit_ids)
    return {
        "batch_id": f"{parent_batch['batch_id']}_{suffix}",
        "unit_ids": unit_ids,
        "source_text": source_text,
        "source_characters": len(source_text),
    }

async def request_translation(page, batch):
    query_url = "https://translate.google.com/?" + urlencode({
        "hl": "fr",
        "sl": "fr",
        "tl": TARGET_LANGUAGE_CODE,
        "text": batch["source_text"],
        "op": "translate",
    })
    await page.goto(query_url, wait_until="domcontentloaded", timeout=60_000)

    deadline = time.monotonic() + TRANSLATION_TIMEOUT_SECONDS
    marker_names = expected_markers(batch)
    last_translated = None
    stable_reads = 0
    while time.monotonic() < deadline:
        page_text = (await page.locator("body").inner_text()).lower()
        protection_terms = ("captcha", "trafic inhabituel", "unusual traffic", "too many requests", "erreur 429")
        if any(term in page_text for term in protection_terms):
            raise ProtectionDetected("Google a affiché une protection ou une limitation de trafic.")

        output = page.locator(f'[lang="{TARGET_LANGUAGE_CODE}"][jsname="jqKxS"]').first
        if await output.count() and await output.is_visible():
            translated = (await output.inner_text()).strip()
            if translated:
                stable_reads = stable_reads + 1 if translated == last_translated else 0
                last_translated = translated
                missing_markers = [marker for marker in marker_names if marker not in translated]
                if not missing_markers:
                    return translated, []
                if stable_reads >= 3:
                    return translated, missing_markers
        await page.wait_for_timeout(750)
    diagnostic_path = STEP_DIR / f"diagnostic_{batch['batch_id']}.png"
    await page.screenshot(path=str(diagnostic_path), full_page=True)
    raise TimeoutError(f"Aucun résultat de traduction récupéré. Capture : {diagnostic_path}")

async def translate_with_fallback(page, batch, depth=0):
    translated, missing_markers = await request_translation(page, batch)
    if not missing_markers:
        return translated

    if len(batch["unit_ids"]) == 1:
        marker = expected_markers(batch)[0]
        print(f"    ↳ identifiant {marker} restauré sur une unité isolée")
        return f"{marker} {translated}"

    middle = len(batch["unit_ids"]) // 2
    left = make_sub_batch(batch, batch["unit_ids"][:middle], f"{depth}L")
    right = make_sub_batch(batch, batch["unit_ids"][middle:], f"{depth}R")
    print(
        f"    ↳ {len(missing_markers)} identifiant(s) perdu(s) ; "
        f"repli en sous-lots de {len(left['unit_ids'])} et {len(right['unit_ids'])} unités"
    )
    left_translation = await translate_with_fallback(page, left, depth + 1)
    await page.wait_for_timeout(int(DELAY_SECONDS * 1000))
    right_translation = await translate_with_fallback(page, right, depth + 1)
    return left_translation.rstrip() + "\n" + right_translation.lstrip()

def parse_and_validate_results(verbose=True):
    marker_pattern = re.compile(
        r"^[ \t]*(\d{8}):[ \t]*(.*?)(?=^[ \t]*\d{8}:|\Z)",
        flags=re.DOTALL | re.MULTILINE,
    )
    batch_by_id = {batch["batch_id"]: batch for batch in batches}
    raw_results = read_jsonl(RESULTS_PATH) if RESULTS_PATH.exists() else []
    result_by_batch = {}
    validation_errors = []

    for result in raw_results:
        if result.get("status") != "ok":
            continue
        batch_id = result.get("batch_id")
        batch = batch_by_id.get(batch_id)
        if batch is None:
            validation_errors.append({"batch_id": batch_id, "error": "unknown_batch"})
            continue
        if result.get("source_sha256") != batch["source_sha256"]:
            validation_errors.append({"batch_id": batch_id, "error": "source_sha256_mismatch"})
            continue
        result_by_batch[batch_id] = result

    parsed_parts = {}
    for batch in batches:
        result = result_by_batch.get(batch["batch_id"])
        if result is None:
            continue
        matches = [
            (f"u{numeric_id}", text.strip())
            for numeric_id, text in marker_pattern.findall(result["translated_text"])
        ]
        found_ids = [unit_id for unit_id, _ in matches]
        if found_ids != batch["unit_ids"]:
            validation_errors.append({
                "batch_id": batch["batch_id"],
                "error": "unit_ids_or_order_mismatch",
                "expected": batch["unit_ids"],
                "found": found_ids,
            })
            continue
        if any(not text for _, text in matches):
            validation_errors.append({"batch_id": batch["batch_id"], "error": "empty_translation"})
            continue
        parsed_parts.update(matches)

    grouped = defaultdict(list)
    for unit_id, translated_text in parsed_parts.items():
        unit = unit_by_id[unit_id]
        key = (unit["record_id"], unit["direction"], unit["target_field"])
        grouped[key].append((unit["part_index"], unit["part_count"], translated_text, unit["source_text"]))

    partial_records = []
    for (record_id, direction, target_field), parts in grouped.items():
        parts.sort(key=lambda item: item[0])
        expected_part_count = parts[0][1]
        complete = (
            len(parts) == expected_part_count
            and [item[0] for item in parts] == list(range(expected_part_count))
        )
        translated_text = " ".join(item[2] for item in parts) if complete else None
        source_text = " ".join(item[3] for item in parts) if complete else None
        partial_records.append({
            "record_id": record_id,
            "direction": direction,
            "target_field": target_field,
            "complete": complete,
            "translated_text": translated_text,
            "unchanged_from_source": complete and translated_text == source_text,
            "part_count": expected_part_count,
        })

    write_jsonl_atomic(PARTIAL_TRANSLATIONS_PATH, partial_records)
    report = {
        "validated_at_utc": datetime.now(timezone.utc).isoformat(),
        "completed_batches": len(result_by_batch),
        "total_batches": len(batches),
        "parsed_fragments": len(parsed_parts),
        "reconstructed_fields": len(partial_records),
        "complete_fields": sum(record["complete"] for record in partial_records),
        "unchanged_translations": sum(record["unchanged_from_source"] for record in partial_records),
        "validation_error_count": len(validation_errors),
        "validation_errors": validation_errors,
    }
    temporary_report = VALIDATION_REPORT_PATH.with_suffix(".json.tmp")
    temporary_report.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    temporary_report.replace(VALIDATION_REPORT_PATH)

    if verbose:
        print("--- Contrôle automatique ---")
        print(f"Blocs validés       : {report['completed_batches']:,}/{report['total_batches']:,}")
        print(f"Fragments analysés  : {report['parsed_fragments']:,}")
        print(f"Champs reconstruits : {report['reconstructed_fields']:,}")
        print(f"Erreurs de contrôle : {report['validation_error_count']:,}")
        print("Sortie partielle     :", PARTIAL_TRANSLATIONS_PATH)

    return report

async def run_browser_session():
    from playwright.async_api import async_playwright

    done = completed_batch_ids()
    pending_batches = [batch for batch in batches if batch["batch_id"] not in done]
    initial_report = parse_and_validate_results(verbose=True)
    if initial_report["validation_error_count"]:
        raise RuntimeError("Le contrôle initial contient des erreurs ; traduction continue annulée.")
    if not pending_batches:
        print("Tous les blocs sont déjà terminés.")
        return
    print(f"Progression initiale : {len(done):,}/{len(batches):,} blocs terminés")
    print(f"Exécution continue : {len(pending_batches):,} blocs à traiter")

    async with async_playwright() as manager:
        browser_environment = os.environ.copy()
        if LOCAL_BROWSER_LIB_DIR.exists():
            existing_library_path = browser_environment.get("LD_LIBRARY_PATH", "")
            browser_environment["LD_LIBRARY_PATH"] = str(LOCAL_BROWSER_LIB_DIR) + (":" + existing_library_path if existing_library_path else "")
        launch_options = {"headless": False, "env": browser_environment}
        if BROWSER_CHANNEL:
            launch_options["channel"] = BROWSER_CHANNEL
        browser = await manager.chromium.launch(**launch_options)
        context = await browser.new_context(permissions=["clipboard-read", "clipboard-write"], locale="fr-FR")
        page = await context.new_page()
        await page.goto(
            f"https://translate.google.com/?hl=fr&sl=fr&tl={TARGET_LANGUAGE_CODE}&op=translate",
            wait_until="domcontentloaded",
            timeout=60_000,
        )
        input("Vérifiez Français → Baoulé dans le navigateur, puis appuyez sur Entrée ici… " )
        if INITIAL_DELAY_SECONDS > 0:
            print(f"Décalage initial de {INITIAL_DELAY_SECONDS:.0f} secondes pour lisser les deux navigateurs…")
            await page.wait_for_timeout(int(INITIAL_DELAY_SECONDS * 1000))

        try:
            for position, batch in enumerate(pending_batches, start=1):
                print(f"[{position}/{len(pending_batches)}] {batch['batch_id']} — {batch['source_characters']} caractères")
                translated_text = await translate_with_fallback(page, batch)
                append_jsonl(RESULTS_PATH, {
                    "batch_id": batch["batch_id"],
                    "status": "ok",
                    "translated_text": translated_text,
                    "source_sha256": batch["source_sha256"],
                    "translated_at_utc": datetime.now(timezone.utc).isoformat(),
                    "engine": "google_translate_text_ui",
                    "target_language_code": TARGET_LANGUAGE_CODE,
                })
                print("  ✓ sauvegardé")
                if position % CHECKPOINT_EVERY_BATCHES == 0:
                    checkpoint_report = parse_and_validate_results(verbose=True)
                    if checkpoint_report["validation_error_count"]:
                        raise RuntimeError("Le contrôle automatique a détecté une erreur ; arrêt préventif.")
                    print("✓ Contrôle validé ; poursuite automatique au bloc suivant.")
                await page.wait_for_timeout(int(DELAY_SECONDS * 1000))
        except ProtectionDetected as exc:
            print("ARRÊT DE SÉCURITÉ :", exc)
        except Exception as exc:
            print(f"ARRÊT SUR ERREUR : {type(exc).__name__}: {exc}")
        finally:
            final_report = parse_and_validate_results(verbose=True)
            await browser.close()
            completed_now = completed_batch_ids()
            print(f"Progression sauvegardée : {len(completed_now):,}/{len(batches):,} blocs terminés")
            print(f"Blocs restants : {len(batches) - len(completed_now):,}")

def run_coroutine_in_thread(coroutine):
    state = {}
    def target():
        try:
            state["result"] = asyncio.run(coroutine)
        except BaseException as exc:
            state["error"] = exc
    thread = threading.Thread(target=target)
    thread.start()
    thread.join()
    if "error" in state:
        raise state["error"]
    return state.get("result")

if RUN_BROWSER:
    run_coroutine_in_thread(run_browser_session())
else:
    print("Navigateur non lancé. Passez RUN_BROWSER à True dans la première cellule après avoir lu les consignes.")

--- Contrôle automatique ---
Blocs validés       : 1,088/1,936
Fragments analysés  : 12,463
Champs reconstruits : 12,462
Erreurs de contrôle : 0
Sortie partielle     : /home/vedem_worker/scrapt-bau/data/02b_text_browser/partial_translations.jsonl
Progression initiale : 1,088/1,936 blocs terminés
Exécution continue : 848 blocs à traiter
[1/848] batch_01089 — 3235 caractères
  ✓ sauvegardé
[2/848] batch_01090 — 3368 caractères
  ✓ sauvegardé
[3/848] batch_01091 — 3148 caractères
  ✓ sauvegardé
[4/848] batch_01092 — 3403 caractères
  ✓ sauvegardé
[5/848] batch_01093 — 3266 caractères
  ✓ sauvegardé
[6/848] batch_01094 — 3060 caractères
  ✓ sauvegardé
[7/848] batch_01095 — 3348 caractères
  ✓ sauvegardé
[8/848] batch_01096 — 3075 caractères
  ✓ sauvegardé
[9/848] batch_01097 — 3421 caractères
  ✓ sauvegardé
[10/848] batch_01098 — 3022 caractères
  ✓ sauvegardé
[11/848] batch_01099 — 3419 caractères
  ✓ sauvegardé
[12/848] batch_01100 — 3084 caractères
  ✓ sauvegardé
[13/848] batch_01101 — 

## 5. Relancer manuellement le contrôle (optionnel)

L'étape 4 appelle désormais ce même contrôle automatiquement tous les 100 blocs et à l'arrêt. Cette cellule reste disponible pour reconstruire manuellement les sorties à tout moment, sans lancer le navigateur.

In [ ]:
validation_report = parse_and_validate_results(verbose=True)

## Critères avant toute montée en charge

Le pilote n'est validé que si les dix blocs sont récupérés sans protection, si aucune balise n'est perdue, si les textes ne sont pas tronqués et si un locuteur baoulé juge un échantillon suffisamment fidèle. Le passage à plus de dix blocs doit rester progressif et respecter les conditions du service.